# Phase 2: Models and Training

This notebook covers Person 2 responsibilities:

- Build a CNN from scratch layer by layer
- Build a manual Transfer Learning pipeline with MobileNetV2
- Train both models
- Save the best model weights
- Evaluate on the test split

The target is calorie prediction from Nutrition5k images, so this is a **regression** task. We use MSE loss for training and report MAE, RMSE, and R2 for evaluation.

## 1. Imports and Configuration

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm.auto import tqdm

try:
    from sklearn.metrics import r2_score
except ImportError:
    r2_score = None

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
CNN_EPOCHS = 10
TRANSFER_HEAD_EPOCHS = 5
TRANSFER_FINE_TUNE_EPOCHS = 5

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'notebooks').exists() and (PROJECT_ROOT.parent / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SEARCH_DIRS = [Path.cwd(), PROJECT_ROOT, PROJECT_ROOT / 'data', PROJECT_ROOT / 'notebooks']


def find_existing_file(filename):
    for directory in SEARCH_DIRS:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    return PROJECT_ROOT / filename


def find_existing_dir(dirname):
    for directory in SEARCH_DIRS:
        candidate = directory / dirname
        if candidate.exists():
            return candidate
    return PROJECT_ROOT / dirname

SPLIT_FILES = {
    'train': find_existing_file('train.csv'),
    'val': find_existing_file('val.csv'),
    'test': find_existing_file('test.csv'),
}
IMAGE_DIR = find_existing_dir('images')
MODEL_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root: {PROJECT_ROOT}')
print(f'Using device: {device}')

## 2. Check Preprocessing Outputs

Run `01_data_preparation.ipynb` first. It should create:

- `images/`
- `train.csv`
- `val.csv`
- `test.csv`

In [ ]:
required_paths = [IMAGE_DIR, SPLIT_FILES['train'], SPLIT_FILES['val'], SPLIT_FILES['test']]
missing = [str(path) for path in required_paths if not path.exists()]

if missing:
    raise FileNotFoundError(
        'Missing preprocessing outputs: ' + ', '.join(missing) +
        '. Run 01_data_preparation.ipynb before training.'
    )

print('All preprocessing outputs found.')
print('Images:', IMAGE_DIR)
print('Train split:', SPLIT_FILES['train'])
print('Validation split:', SPLIT_FILES['val'])
print('Test split:', SPLIT_FILES['test'])

## 3. Dataset and DataLoaders

In [ ]:
class NutritionDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.csv_file = Path(csv_file)
        self.data = pd.read_csv(self.csv_file)
        self.transform = transform

        expected_columns = {'image_path', 'calories'}
        if not expected_columns.issubset(self.data.columns):
            raise ValueError(
                f'{csv_file} must contain columns {expected_columns}. '
                f'Found: {list(self.data.columns)}'
            )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = Path(row['image_path'])
        if not image_path.is_absolute() and not image_path.exists():
            image_path = IMAGE_DIR / image_path.name
        image = Image.open(image_path).convert('RGB')
        calories = torch.tensor([float(row['calories'])], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, calories

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = NutritionDataset(SPLIT_FILES['train'], transform=train_transforms)
val_dataset = NutritionDataset(SPLIT_FILES['val'], transform=val_test_transforms)
test_dataset = NutritionDataset(SPLIT_FILES['test'], transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train images: {len(train_dataset)}')
print(f'Validation images: {len(val_dataset)}')
print(f'Test images: {len(test_dataset)}')

## 4. CNN From Scratch

The architecture is built manually with convolution, ReLU, max pooling, batch normalization, flatten, dense, dropout, and one regression output neuron.

In [ ]:
class ScratchCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        return self.regressor(x)

scratch_cnn = ScratchCNN().to(device)
print(scratch_cnn)

## 5. Transfer Learning: MobileNetV2 Built Manually

We load MobileNetV2 without using its original classifier as the final solution, replace it with our own explicit regression head, freeze the backbone first, then unfreeze the last feature blocks for fine-tuning.

In [ ]:
def build_mobilenet_v2_regressor(freeze_backbone=True):
    weights = models.MobileNet_V2_Weights.DEFAULT
    model = models.mobilenet_v2(weights=weights)

    for param in model.features.parameters():
        param.requires_grad = not freeze_backbone

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, 1),
    )
    return model

mobilenet_v2 = build_mobilenet_v2_regressor(freeze_backbone=True).to(device)
print(mobilenet_v2.classifier)

## 6. Training and Evaluation Helpers

In [ ]:
def count_trainable_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def run_one_epoch(model, data_loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()
    total_loss = 0.0

    context = torch.enable_grad() if is_training else torch.no_grad()
    with context:
        for images, targets in tqdm(data_loader, leave=False):
            images = images.to(device)
            targets = targets.to(device)

            if is_training:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, targets)

            if is_training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)

    return total_loss / len(data_loader.dataset)


def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, save_path):
    history = []
    best_val_loss = float('inf')
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        train_loss = run_one_epoch(model, train_loader, criterion, optimizer)
        val_loss = run_one_epoch(model, val_loader, criterion, optimizer=None)

        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
        print(f'Epoch {epoch:02d}/{epochs} | train MSE: {train_loss:.2f} | val MSE: {val_loss:.2f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f'  Saved best weights: {save_path}')

    training_time = time.time() - start_time
    return pd.DataFrame(history), training_time


def predict(model, data_loader):
    model.eval()
    predictions = []
    targets = []

    with torch.no_grad():
        for images, y in tqdm(data_loader, leave=False):
            images = images.to(device)
            outputs = model(images).cpu().numpy().reshape(-1)
            predictions.extend(outputs.tolist())
            targets.extend(y.numpy().reshape(-1).tolist())

    return np.array(targets), np.array(predictions)


def evaluate_regression(model, data_loader):
    y_true, y_pred = predict(model, data_loader)
    mse = float(np.mean((y_true - y_pred) ** 2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(y_true - y_pred)))

    if r2_score is not None:
        r2 = float(r2_score(y_true, y_pred))
    else:
        ss_res = float(np.sum((y_true - y_pred) ** 2))
        ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
        r2 = 1 - ss_res / ss_tot if ss_tot else 0.0

    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

## 7. Train CNN From Scratch

In [ ]:
scratch_cnn = ScratchCNN().to(device)
criterion = nn.MSELoss()
optimizer_cnn = optim.Adam(scratch_cnn.parameters(), lr=1e-3)

print(f'Trainable parameters: {count_trainable_parameters(scratch_cnn):,}')
cnn_history, cnn_time = train_model(
    scratch_cnn,
    train_loader,
    val_loader,
    criterion,
    optimizer_cnn,
    epochs=CNN_EPOCHS,
    save_path=MODEL_DIR / 'cnn_from_scratch.pth',
)
cnn_history.to_csv(RESULTS_DIR / 'cnn_training_history.csv', index=False)

## 8. Train MobileNetV2 Head, Then Fine-Tune

In [ ]:
mobilenet_v2 = build_mobilenet_v2_regressor(freeze_backbone=True).to(device)
criterion = nn.MSELoss()

print('Phase 1: train custom regression head with frozen backbone')
print(f'Trainable parameters: {count_trainable_parameters(mobilenet_v2):,}')
optimizer_head = optim.Adam(filter(lambda p: p.requires_grad, mobilenet_v2.parameters()), lr=1e-3)
head_history, head_time = train_model(
    mobilenet_v2,
    train_loader,
    val_loader,
    criterion,
    optimizer_head,
    epochs=TRANSFER_HEAD_EPOCHS,
    save_path=MODEL_DIR / 'mobilenet_v2_head_best.pth',
)

print('Phase 2: unfreeze last MobileNetV2 feature blocks for fine-tuning')
for param in mobilenet_v2.features[-4:].parameters():
    param.requires_grad = True

print(f'Trainable parameters after unfreezing: {count_trainable_parameters(mobilenet_v2):,}')
optimizer_fine = optim.Adam(filter(lambda p: p.requires_grad, mobilenet_v2.parameters()), lr=1e-5)
fine_history, fine_time = train_model(
    mobilenet_v2,
    train_loader,
    val_loader,
    criterion,
    optimizer_fine,
    epochs=TRANSFER_FINE_TUNE_EPOCHS,
    save_path=MODEL_DIR / 'mobilenet_v2_finetuned.pth',
)

mobilenet_history = pd.concat([
    head_history.assign(phase='frozen_backbone'),
    fine_history.assign(phase='fine_tuning'),
], ignore_index=True)
mobilenet_history.to_csv(RESULTS_DIR / 'mobilenet_v2_training_history.csv', index=False)
mobilenet_time = head_time + fine_time

## 9. Test Evaluation and Model Comparison

In [ ]:
scratch_cnn.load_state_dict(torch.load(MODEL_DIR / 'cnn_from_scratch.pth', map_location=device))
mobilenet_v2.load_state_dict(torch.load(MODEL_DIR / 'mobilenet_v2_finetuned.pth', map_location=device))

cnn_metrics = evaluate_regression(scratch_cnn, test_loader)
mobilenet_metrics = evaluate_regression(mobilenet_v2, test_loader)

comparison = pd.DataFrame([
    {
        'model': 'CNN from scratch',
        'trainable_parameters': count_trainable_parameters(scratch_cnn),
        'training_time_seconds': round(cnn_time, 2),
        **cnn_metrics,
    },
    {
        'model': 'MobileNetV2 transfer learning',
        'trainable_parameters': count_trainable_parameters(mobilenet_v2),
        'training_time_seconds': round(mobilenet_time, 2),
        **mobilenet_metrics,
    },
])

comparison.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)
comparison

## 10. Save Final Full Models

The `.pth` files above save weights. These `.pt` files save the full PyTorch model objects for easier loading in demos.

In [ ]:
torch.save(scratch_cnn, MODEL_DIR / 'cnn_from_scratch_full.pt')
torch.save(mobilenet_v2, MODEL_DIR / 'mobilenet_v2_full.pt')
print('Saved final models in:', MODEL_DIR.resolve())